# VAR Simulation Pipeline — Library Implementation
Evaluating Scikit-Learn/PyMC shrinkage priors across Scenarios

In [ ]:


import os
os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64"
# CPU required: PyMC NUTS sampler is incompatible with CUDA backend

import numpy as np
import pandas as pd
import time
import os
import matplotlib.pyplot as plt
import warnings
from abc import ABC, abstractmethod
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from scipy import stats
import pymc as pm
from tqdm import tqdm

warnings.filterwarnings('ignore')

N_REPLICATIONS = 5
T_TOTAL = 200
T_TEST = 20
BURNIN = 50
SPARSITY_PROB = 0.3
COEF_RANGE = (-0.4, 0.4)
SIGMA_DIAG = 0.1

LAMBDA_RIDGE = 0.1
CI_ALPHA = 0.05
MCMC_TUNE = 1000
MCMC_DRAWS = 1000
MCMC_CHAINS = 2
SEED = 123

SCENARIOS = {
    1: {'name': 'Low-dim (d=3, p=4)', 'd': 3, 'p_fit': 4, 'seed': 101},
    2: {'name': 'High-dim (d=20, p=1)', 'd': 20, 'p_fit': 1, 'seed': 202},
    3: {'name': 'High-dim overfit (d=20, p=4)', 'd': 20, 'p_fit': 4, 'seed': 303},
}

def gen_var_data(d, T, burnin=BURNIN, rng=None):
    if rng is None: rng = np.random.RandomState()
    A = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            if rng.uniform() > SPARSITY_PROB:
                A[i, j] = rng.uniform(*COEF_RANGE)
    eigvals = np.linalg.eigvals(A)
    max_mod = np.max(np.abs(eigvals))
    if max_mod >= 1.0: A = A / (1.1 * max_mod)
    Sigma = SIGMA_DIAG * np.eye(d)
    Y = np.zeros((T + burnin, d))
    Y[0, :] = rng.randn(d)
    for t in range(1, T + burnin):
        Y[t, :] = rng.multivariate_normal(A @ Y[t - 1, :], Sigma)
    return {'Y': Y[burnin:, :], 'A_true': A, 'Sigma_true': Sigma}

def make_var_design(Y, p):
    T, d = Y.shape
    X = np.zeros((T - p, d * p))
    Y_out = Y[p:, :]
    for t in range(p, T):
        lags = []
        for lag_i in range(1, p + 1):
            lags.extend(Y[t - lag_i, :])
        X[t - p, :] = lags
    return X, Y_out

def get_true_B(A_true, d, p_fit):
    B_true = np.zeros((d, d * p_fit))
    B_true[:, :d] = A_true
    return B_true



In [ ]:
class VARModel(ABC):
    @property
    @abstractmethod
    def name(self): pass
    @abstractmethod
    def fit(self, X_train, Y_train, d, p_fit): pass
    @abstractmethod
    def get_coefficients(self): pass
    @abstractmethod
    def get_intervals(self, alpha=0.05): pass

class SklearnRidgeVAR(VARModel):
    def __init__(self, alpha=LAMBDA_RIDGE):
        self.alpha = alpha
        self._name = 'SklearnRidge'
    @property
    def name(self): return self._name
    def fit(self, X_train, Y_train, d, p_fit):
        self.model = MultiOutputRegressor(Ridge(alpha=self.alpha, fit_intercept=False))
        self.model.fit(X_train, Y_train)
        self.B_hat_ = np.vstack([est.coef_ for est in self.model.estimators_])
        # Analytic approx for fast runtime
        resid = Y_train - self.model.predict(X_train)
        sigma2 = np.var(resid, axis=0)
        XtX_inv = np.linalg.pinv(X_train.T @ X_train)
        se = np.sqrt(np.clip(np.outer(sigma2, np.diag(XtX_inv)), 0, None))
        z_val = stats.norm.ppf(1 - CI_ALPHA / 2)
        self.B_lower_ = self.B_hat_ - z_val * se
        self.B_upper_ = self.B_hat_ + z_val * se
        return self
    def get_coefficients(self): return self.B_hat_
    def get_intervals(self, alpha=0.05): return self.B_lower_, self.B_upper_

class SklearnOLSVAR(VARModel):
    def __init__(self):
        self._name = 'SklearnOLS'
    @property
    def name(self): return self._name
    def fit(self, X_train, Y_train, d, p_fit):
        self.model = MultiOutputRegressor(LinearRegression(fit_intercept=False))
        self.model.fit(X_train, Y_train)
        self.B_hat_ = np.vstack([est.coef_ for est in self.model.estimators_])
        resid = Y_train - self.model.predict(X_train)
        sigma2 = np.var(resid, axis=0)
        XtX_inv = np.linalg.pinv(X_train.T @ X_train)
        se = np.sqrt(np.clip(np.outer(sigma2, np.diag(XtX_inv)), 0, None))
        z_val = stats.norm.ppf(1 - CI_ALPHA / 2)
        self.B_lower_ = self.B_hat_ - z_val * se
        self.B_upper_ = self.B_hat_ + z_val * se
        return self
    def get_coefficients(self): return self.B_hat_
    def get_intervals(self, alpha=0.05): return self.B_lower_, self.B_upper_

class PyMCBayesianVAR(VARModel):
    def __init__(self, prior_type, prior_scale=1.0, tune=MCMC_TUNE, draws=MCMC_DRAWS, chains=MCMC_CHAINS):
        self.prior_type = prior_type
        self.prior_scale = prior_scale
        self.tune = tune
        self.draws = draws
        self.chains = chains
        self._name = f'PyMC_{prior_type}'
    @property
    def name(self): return self._name

    def fit(self, X_train, Y_train, d, p_fit):
        q = X_train.shape[1]
        with pm.Model() as model:
            if self.prior_type == "Normal":
                B = pm.Normal("B", mu=0, sigma=self.prior_scale, shape=(d, q))
            elif self.prior_type == "Lasso":
                B = pm.Laplace("B", mu=0, b=1.0/self.prior_scale, shape=(d, q))
            elif self.prior_type == "Horseshoe":
                tau = pm.HalfCauchy("tau", beta=1, shape=(d, 1))
                lambd = pm.HalfCauchy("lambda", beta=1, shape=(d, q))
                B = pm.Normal("B", mu=0, sigma=tau * lambd, shape=(d, q))
            elif self.prior_type == "SpikeSlab":
                pi = pm.Beta("pi", 1, 1)
                sigma_slab = pm.HalfNormal("sigma_slab", sigma=1)
                slab = pm.Normal.dist(mu=0, sigma=sigma_slab)
                spike = pm.Normal.dist(mu=0, sigma=1e-4)
                w = pm.math.stack([pi, 1-pi])
                components = [slab, spike]
                B = pm.Mixture("B", w=w, comp_dists=components, shape=(d, q))

            sigma = pm.HalfCauchy("sigma", beta=2.5, shape=d)
            mu = pm.math.dot(X_train, B.T)
            Y_obs = pm.Normal("Y_obs", mu=mu, sigma=sigma, observed=Y_train)
            import logging
            logger = logging.getLogger("pymc")
            logger.setLevel(logging.ERROR)
            trace = pm.sample(draws=self.draws, tune=self.tune, chains=self.chains, 
                              target_accept=0.85, random_seed=SEED, progressbar=True, compute_convergence_checks=False)

        self.B_samples_ = trace.posterior["B"].values.reshape(-1, d, q)
        self.B_hat_ = np.mean(self.B_samples_, axis=0)
        self.B_lower_ = np.percentile(self.B_samples_, 2.5, axis=0)
        self.B_upper_ = np.percentile(self.B_samples_, 97.5, axis=0)
        return self

    def get_coefficients(self): return self.B_hat_
    def get_intervals(self, alpha=0.05): return self.B_lower_, self.B_upper_



In [ ]:
MODEL_REGISTRY = {
    'OLS': lambda: SklearnOLSVAR(),
    'Ridge': lambda: SklearnRidgeVAR(),
    'Normal': lambda: PyMCBayesianVAR('Normal'),
    'Lasso': lambda: PyMCBayesianVAR('Lasso'),
    'Horseshoe': lambda: PyMCBayesianVAR('Horseshoe'),
    'Spike and Slab': lambda: PyMCBayesianVAR('SpikeSlab'),
}

def compute_aic_bic(Y_train, pred_train, d, p):
    T = Y_train.shape[0]
    eps = Y_train - pred_train
    Sigma = eps.T @ eps / T
    Sigma += np.eye(d) * 1e-8
    det_S = np.linalg.det(Sigma)
    if det_S <= 0: det_S = 1e-8
    log_det = np.log(det_S)
    k = d * d * p
    return float(T * log_det + 2 * k), float(T * log_det + np.log(T) * k)

class MetricsEngine:
    @staticmethod
    def forecast_rmse(B, Ytrain, Ytest, p):
        Y_for_forecast = np.vstack([Ytrain[-p:, :], Ytest])
        Ttest = Ytest.shape[0]
        rmse_vec = []
        for i in tqdm(range(p, p + Ttest - 1), desc="Simulated Forecast", leave=False, ascii=True, ncols=80):
            lags = []
            for lag_j in range(p): lags.extend(Y_for_forecast[i - lag_j, :])
            pred = np.array(lags).reshape(1, -1) @ B.T
            actual = Y_for_forecast[i + 1, :]
            rmse_vec.append(np.sqrt(np.mean((actual - pred.flatten())**2)))
        return np.mean(rmse_vec)
    
    @staticmethod
    def param_metrics(B_est, B_lower, B_upper, B_true, tol=1e-3):
        b_e = B_est.flatten()
        b_lo = B_lower.flatten()
        b_hi = B_upper.flatten()
        b_t = B_true.flatten()
        
        zero_mask = np.abs(b_t) < tol
        nz_mask = ~zero_mask
        
        metrics = {}
        # Overall
        metrics['param_rmse'] = float(np.sqrt(np.mean((b_e - b_t)**2)))
        metrics['coverage'] = float(np.mean((b_t >= b_lo) & (b_t <= b_hi)))
        metrics['int_length'] = float(np.mean(b_hi - b_lo))
        
        # Zero
        if np.sum(zero_mask) > 0:
            metrics['param_rmse_zero'] = float(np.sqrt(np.mean((b_e[zero_mask] - b_t[zero_mask])**2)))
            metrics['coverage_zero'] = float(np.mean((b_t[zero_mask] >= b_lo[zero_mask]) & (b_t[zero_mask] <= b_hi[zero_mask])))
            metrics['int_length_zero'] = float(np.mean(b_hi[zero_mask] - b_lo[zero_mask]))
        else:
            metrics['param_rmse_zero'] = np.nan
            metrics['coverage_zero'] = np.nan
            metrics['int_length_zero'] = np.nan
            
        # NonZero
        if np.sum(nz_mask) > 0:
            metrics['param_rmse_nonzero'] = float(np.sqrt(np.mean((b_e[nz_mask] - b_t[nz_mask])**2)))
            metrics['coverage_nonzero'] = float(np.mean((b_t[nz_mask] >= b_lo[nz_mask]) & (b_t[nz_mask] <= b_hi[nz_mask])))
            metrics['int_length_nonzero'] = float(np.mean(b_hi[nz_mask] - b_lo[nz_mask]))
        else:
            metrics['param_rmse_nonzero'] = np.nan
            metrics['coverage_nonzero'] = np.nan
            metrics['int_length_nonzero'] = np.nan
            
        # Sparsity metrics
        is_zero_est = np.abs(b_e) < tol
        TP_zero = np.sum(zero_mask & is_zero_est)
        FN_zero = np.sum(zero_mask & ~is_zero_est)
        metrics['sparsity_recovery_rate'] = float(TP_zero / max(1, (TP_zero + FN_zero)))

        is_nz_est = np.abs(b_e) >= tol
        FP_nz = np.sum(zero_mask & is_nz_est)
        TP_nz = np.sum(nz_mask & is_nz_est)
        metrics['false_discovery_rate'] = float(FP_nz / max(1, (FP_nz + TP_nz)))
        
        return metrics

def run_single_replication(scen_id, rep, rng):
    cfg = SCENARIOS[scen_id]
    d, p_fit = cfg['d'], cfg['p_fit']
    gen_out = gen_var_data(d, T_TOTAL, burnin=BURNIN, rng=rng)
    Yfull = gen_out['Y']
    Ttrain = T_TOTAL - T_TEST
    Ytrain, Ytest = Yfull[:Ttrain, :], Yfull[Ttrain:, :]
    X_train, Y_train = make_var_design(Ytrain, p_fit)
    B_true = get_true_B(gen_out['A_true'], d, p_fit)
    
    results = []
    
    # Compute OLS first to calculate Shrinkage Ratio
    ols_model = SklearnOLSVAR().fit(X_train, Y_train, d, p_fit)
    B_ols = ols_model.get_coefficients()
    mean_abs_B_ols = np.mean(np.abs(B_ols))
    
    for name, factory in MODEL_REGISTRY.items():
        if name == 'OLS':
            model = ols_model
            B_hat = B_ols
            B_lo, B_hi = model.get_intervals()
        else:
            try:
                model = factory().fit(X_train, Y_train, d, p_fit)
                B_hat = model.get_coefficients()
                B_lo, B_hi = model.get_intervals()
            except Exception as e:
                print(f"    [{name}] FAILED: {e} — skipping")
                continue
            
        pred_train = X_train @ B_hat.T
        aic, bic = compute_aic_bic(Y_train, pred_train, d, p_fit)
        f_rmse = MetricsEngine.forecast_rmse(B_hat, Ytrain, Ytest, p_fit)
        mets = MetricsEngine.param_metrics(B_hat, B_lo, B_hi, B_true)
        shrinkage = np.mean(np.abs(B_hat)) / max(1e-8, mean_abs_B_ols)
        
        row = {
            'scenario': scen_id, 'scenario_name': SCENARIOS[scen_id]['name'],
            'rep': rep, 'model': name,
            'forecast_rmse': f_rmse, 'shrinkage_ratio': shrinkage,
            'AIC': aic, 'BIC': bic
        }
        row.update(mets)
        results.append(row)
    return results

all_results = []
for s_id in SCENARIOS.keys():
    rng = np.random.RandomState(SCENARIOS[s_id]['seed'])
    scen_name = SCENARIOS[s_id]['name']
    print(f"Running Scenario {s_id}: {scen_name}")
    for r in tqdm(range(1, N_REPLICATIONS + 1), desc="Replications", ascii=True, ncols=80):
        t0 = time.time()
        res = run_single_replication(s_id, r, rng)
        all_results.extend(res)
        elapsed = round(time.time() - t0, 1)
        print(f"  Rep {r}/{N_REPLICATIONS} Done in {elapsed}s")

df = pd.DataFrame(all_results)

# --- Save results ---
OUTPUT_DIR = os.path.join(os.getcwd(), 'simulation_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)
csv_path = os.path.join(OUTPUT_DIR, 'simulation_metrics.csv')
df.to_csv(csv_path, index=False)
print("Results saved to", csv_path)

df.head(10)



In [ ]:
# ============================================================
# VISUALIZATION — Metric comparison across Models x Scenarios
# ============================================================
import seaborn as sns

if len(df) > 0:
    df_avg = df.groupby(['scenario_name', 'model'])[['forecast_rmse', 'param_rmse', 'coverage',
                                                       'sparsity_recovery_rate', 'false_discovery_rate']].mean().reset_index()

    scenarios = df_avg['scenario_name'].unique()
    n_scen = len(scenarios)
    fig, axes = plt.subplots(n_scen, 2, figsize=(16, 5 * n_scen))
    if n_scen == 1: axes = axes.reshape(1, -1)

    for idx, scen in enumerate(scenarios):
        scen_df = df_avg[df_avg['scenario_name'] == scen].sort_values('forecast_rmse')
        colors = ['#2196F3' if 'PyMC' not in m else '#FF5722' for m in scen_df['model']]

        ax0 = axes[idx, 0]
        ax0.barh(scen_df['model'], scen_df['forecast_rmse'], color=colors)
        ax0.set_title(f'{scen}\nForecast RMSE (lower=better)', fontsize=11)
        ax0.set_xlabel('Forecast RMSE')
        ax0.grid(axis='x', alpha=0.3)

        ax1 = axes[idx, 1]
        ax1.barh(scen_df['model'], scen_df['param_rmse'], color=colors)
        ax1.set_title(f'{scen}\nParam RMSE vs B_true (lower=better)', fontsize=11)
        ax1.set_xlabel('Param RMSE')
        ax1.grid(axis='x', alpha=0.3)

    from matplotlib.patches import Patch
    handles = [Patch(color='#2196F3', label='Frequentist'), Patch(color='#FF5722', label='Bayesian')]
    fig.legend(handles=handles, loc='upper right', fontsize=11)
    plt.suptitle('VAR Simulation — Shrinkage Prior Comparison', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'simulation_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Simulation comparison chart saved.")
else:
    print("No results to plot.")



In [ ]:
# ============================================================
# VISUALIZATION — Sparsity Recovery and CI Coverage heatmap
# ============================================================
if len(df) > 0:
    df_avg = df.groupby(['scenario_name', 'model'])[['coverage', 'sparsity_recovery_rate',
                                                       'false_discovery_rate', 'shrinkage_ratio']].mean().reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    piv_cov = df_avg.pivot(index='model', columns='scenario_name', values='coverage')
    sns.heatmap(piv_cov, annot=True, fmt='.2f', cmap='Greens', ax=axes[0],
                linewidths=0.5, vmin=0, vmax=1)
    axes[0].set_title('CI Coverage Rate (target=0.95)', fontsize=12, fontweight='bold')

    piv_spar = df_avg.pivot(index='model', columns='scenario_name', values='sparsity_recovery_rate')
    sns.heatmap(piv_spar, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
                linewidths=0.5, vmin=0, vmax=1)
    axes[1].set_title('Sparsity Recovery Rate (higher=better)', fontsize=12, fontweight='bold')

    plt.suptitle('Uncertainty Quantification — Coverage & Sparsity', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'simulation_coverage_sparsity.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Coverage/sparsity chart saved.")

